# Synthetic Maritime Risk Dataset Generator

This notebook generates synthetic maritime shipment data for logistics risk simulations and scenario analysis.

## Contents

1. Imports and setup
2. Helper functions
3. Shipment generator
4. Scenario parameters and dataset creation
5. Diagnostics
6. Export to CSV

## Dataset outputs

The generated dataset includes route, ETA, delay, geopolitical event, risk, and freight cost fields suitable for analytics and machine learning experiments.

## Column meanings

- **shipment_id**: Unique shipment identifier.
- **departure_timestamp**: Simulated departure timestamp.
- **origin_port**: Origin port name.
- **destination_port**: Destination port name.
- **origin_region**: Origin macro-region.
- **destination_region**: Destination macro-region.
- **route_corridor**: Assigned logistics corridor, such as Suez Corridor or Pacific Crossing.
- **ocean_side**: Route profile across ocean sides.
- **vessel_type**: Vessel category, such as Container, Bulk Carrier, or Tanker.
- **cargo_type**: Cargo category.
- **distance_nm**: Estimated route distance in nautical miles.
- **avg_speed_knots**: Simulated average vessel speed in knots.
- **baseline_eta_days**: Baseline ETA in days without disruptions.
- **weather_delay_days**: Delay caused by weather conditions.
- **port_handling_days**: Delay caused by port operations.
- **geopolitical_event**: Event applied to the shipment, or `None` when no event occurs.
- **geopolitical_delay_days**: Delay specifically caused by geopolitical disruption.
- **expected_delay_days**: Total expected delay from weather, port handling, and geopolitical effects.
- **adjusted_eta_days**: Final ETA after all simulated delays.
- **risk_score**: Synthetic normalized risk metric from 0 to 1.
- **freight_cost_index**: Synthetic freight cost multiplier.
- **baseline_arrival_timestamp**: Baseline arrival timestamp.
- **adjusted_arrival_timestamp**: Arrival timestamp after considering disruptions.
- **delay_class**: Delay category, such as `on_time`, `delayed`, or `critical_delay`.

## Prediction possibilities

This dataset supports several predictive tasks:

1. **Regression**
- Predict `expected_delay_days`.
- Predict `adjusted_eta_days`.
- Predict `freight_cost_index`.

2. **Classification**
- Predict `delay_class`.
- Create and predict SLA breach flags, for example `is_late = expected_delay_days > 2`.
- Create and predict high-risk flags, for example `risk_score >= 0.7`.

3. **Scenario simulation and what-if analysis**
- Compare outputs with events enabled versus disabled.
- Quantify sensitivity by corridor, such as Suez, Panama, Hormuz, and Malacca.
- Rank routes by disruption impact.

## Notes

- The data is synthetic and intended for prototyping and model experimentation.
- You can tune the event list, event intensity, number of rows, and route rules to create multiple scenarios.

## 1. Imports and Setup

This section loads all required libraries, sets the random seed, and defines the base reference dictionaries used throughout the notebook.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

np.random.seed(42)

PORTS = {
    "Shanghai": {"country": "China", "lat": 31.2304, "lon": 121.4737, "region": "East Asia"},
    "Singapore": {"country": "Singapore", "lat": 1.3521, "lon": 103.8198, "region": "Southeast Asia"},
    "Rotterdam": {"country": "Netherlands", "lat": 51.9244, "lon": 4.4777, "region": "Europe"},
    "Santos": {"country": "Brazil", "lat": -23.9608, "lon": -46.3336, "region": "South America"},
    "Los Angeles": {"country": "USA", "lat": 33.7405, "lon": -118.2719, "region": "North America"},
    "Houston": {"country": "USA", "lat": 29.7604, "lon": -95.3698, "region": "North America"},
    "Dubai": {"country": "UAE", "lat": 25.2048, "lon": 55.2708, "region": "Middle East"},
    "Mumbai": {"country": "India", "lat": 19.0760, "lon": 72.8777, "region": "South Asia"},
    "Durban": {"country": "South Africa", "lat": -29.8587, "lon": 31.0218, "region": "Africa"},
    "Sydney": {"country": "Australia", "lat": -33.8688, "lon": 151.2093, "region": "Oceania"},
}

VESSEL_TYPES = ["Container", "Bulk Carrier", "Tanker"]
CARGO_TYPES = ["Electronics", "Machinery", "Food", "Chemicals", "Textiles", "Auto Parts"]

SPEED_RANGES = {
    "Container": (15, 23),
    "Bulk Carrier": (11, 16),
    "Tanker": (12, 18),
}

GEO_EVENT_LIBRARY = {
    "Red Sea Tension": {"delay_days": 2.5, "cost_mult": 1.10, "risk_add": 0.18},
    "Suez Disruption": {"delay_days": 4.0, "cost_mult": 1.16, "risk_add": 0.24},
    "Panama Drought": {"delay_days": 3.5, "cost_mult": 1.12, "risk_add": 0.19},
    "Port Strike": {"delay_days": 2.0, "cost_mult": 1.08, "risk_add": 0.14},
    "Customs Slowdown": {"delay_days": 1.2, "cost_mult": 1.05, "risk_add": 0.09},
    "Hormuz Tension": {"delay_days": 3.2, "cost_mult": 1.14, "risk_add": 0.22},
    "Strait of Malacca Tension": {"delay_days": 2.8, "cost_mult": 1.11, "risk_add": 0.17},
}

print(f"Loaded {len(PORTS)} ports and {len(GEO_EVENT_LIBRARY)} geopolitical event templates.")

Loaded 10 ports and 7 geopolitical event templates.


## 2. Helper Functions

This section defines the geographic and scenario helper functions used to infer route behavior and simulate geopolitical disruption effects.

In [2]:
def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0
    p1 = np.radians(lat1)
    p2 = np.radians(lat2)
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)

    a = np.sin(dlat / 2.0) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return r * c


def infer_route_profile(origin, destination):
    # Simple deterministic logic for corridor and ocean side
    regions = {PORTS[origin]["region"], PORTS[destination]["region"]}

    if {"East Asia", "Europe"}.issubset(regions) or {"Southeast Asia", "Europe"}.issubset(regions):
        return "Suez Corridor", "Indian-Atlantic Side", 1.08
    if {"East Asia", "North America"}.issubset(regions) or {"Southeast Asia", "North America"}.issubset(regions):
        return "Pacific Crossing", "Pacific Side", 1.02
    if {"Europe", "South America"}.issubset(regions) or {"Europe", "North America"}.issubset(regions):
        return "Atlantic Crossing", "Atlantic Side", 1.00
    if {"South Asia", "Europe"}.issubset(regions) or {"Middle East", "Europe"}.issubset(regions):
        return "Suez Corridor", "Indian-Atlantic Side", 1.06
    if {"South America", "North America"}.issubset(regions):
        return "Panama Corridor", "Atlantic-Pacific Side", 1.10
    return "Cape Diversion", "Indian-Atlantic Side", 1.14


def draw_geopolitical_event(enable_events, active_events, corridor):
    if not enable_events:
        return "None", 0.0, 1.0, 0.0

    base_prob = 0.20
    corridor_bias = 0.08 if corridor in {"Suez Corridor", "Panama Corridor"} else 0.0
    trigger = np.random.rand() < (base_prob + corridor_bias)

    if not trigger:
        return "None", 0.0, 1.0, 0.0

    event_name = np.random.choice(active_events) if active_events else "None"
    if event_name == "None":
        return "None", 0.0, 1.0, 0.0

    event_cfg = GEO_EVENT_LIBRARY[event_name]
    severity = np.round(np.random.uniform(0.4, 1.0), 2)
    delay_days = event_cfg["delay_days"] * severity
    cost_mult = 1 + (event_cfg["cost_mult"] - 1) * severity
    risk_add = event_cfg["risk_add"] * severity
    return event_name, delay_days, cost_mult, risk_add

## 3. Shipment Generator

This section contains the main generator that creates synthetic shipment records, computes ETA values, assigns delays, and builds the final dataframe.

In [3]:
def generate_synthetic_shipments(
    n_rows=5000,
    start_date="2025-01-01",
    end_date="2025-12-31",
    enable_geopolitical_events=True,
    active_events=None,
):
    if active_events is None:
        active_events = list(GEO_EVENT_LIBRARY.keys())

    rows = []
    port_names = list(PORTS.keys())

    departure_dates = pd.to_datetime(
        np.random.choice(pd.date_range(start_date, end_date, freq="h"), size=n_rows)
    )

    for i in range(n_rows):
        origin = np.random.choice(port_names)
        destination = np.random.choice([p for p in port_names if p != origin])

        vessel_type = np.random.choice(VESSEL_TYPES, p=[0.55, 0.25, 0.20])
        cargo_type = np.random.choice(CARGO_TYPES)

        corridor, ocean_side, route_factor = infer_route_profile(origin, destination)

        o = PORTS[origin]
        d = PORTS[destination]
        distance_km = haversine_km(o["lat"], o["lon"], d["lat"], d["lon"]) * route_factor
        distance_nm = distance_km * 0.539957

        speed_min, speed_max = SPEED_RANGES[vessel_type]
        speed_knots = np.random.uniform(speed_min, speed_max)

        baseline_eta_days = distance_nm / (speed_knots * 24.0)

        port_handling_days = np.random.uniform(0.4, 1.8)
        weather_delay_days = max(0.0, np.random.normal(0.5, 0.6))

        event_name, geo_delay_days, cost_mult_event, risk_add = draw_geopolitical_event(
            enable_geopolitical_events, active_events, corridor
        )

        expected_delay_days = port_handling_days + weather_delay_days + geo_delay_days
        adjusted_eta_days = baseline_eta_days + expected_delay_days

        delay_class = (
            "on_time"
            if expected_delay_days < 1.5
            else "delayed" if expected_delay_days < 4.0 else "critical_delay"
        )

        risk_score = min(1.0, np.round(0.15 + risk_add + expected_delay_days / 12.0, 3))
        freight_cost_index = np.round(1.0 * cost_mult_event * (1 + expected_delay_days * 0.02), 3)

        departure_ts = departure_dates[i]
        eta_baseline_ts = departure_ts + pd.to_timedelta(baseline_eta_days, unit="D")
        eta_adjusted_ts = departure_ts + pd.to_timedelta(adjusted_eta_days, unit="D")

        rows.append(
            {
                "shipment_id": f"SHP-{i+1:07d}",
                "departure_timestamp": departure_ts,
                "origin_port": origin,
                "destination_port": destination,
                "origin_region": o["region"],
                "destination_region": d["region"],
                "route_corridor": corridor,
                "ocean_side": ocean_side,
                "vessel_type": vessel_type,
                "cargo_type": cargo_type,
                "distance_nm": round(distance_nm, 1),
                "avg_speed_knots": round(speed_knots, 1),
                "baseline_eta_days": round(baseline_eta_days, 2),
                "weather_delay_days": round(weather_delay_days, 2),
                "port_handling_days": round(port_handling_days, 2),
                "geopolitical_event": event_name,
                "geopolitical_delay_days": round(geo_delay_days, 2),
                "expected_delay_days": round(expected_delay_days, 2),
                "adjusted_eta_days": round(adjusted_eta_days, 2),
                "risk_score": risk_score,
                "freight_cost_index": freight_cost_index,
                "baseline_arrival_timestamp": eta_baseline_ts,
                "adjusted_arrival_timestamp": eta_adjusted_ts,
                "delay_class": delay_class,
            }
        )

    return pd.DataFrame(rows)

print("Generator ready.")

Generator ready.


## 4. Scenario Parameters and Dataset Creation

This section exposes the main scenario controls, such as number of rows and active geopolitical events, and then generates the dataset.

In [4]:
# Parameters you can edit
N_ROWS = 12000
ENABLE_GEOPOLITICAL_EVENTS = True
ACTIVE_EVENTS = [
    "Red Sea Tension",
    "Suez Disruption",
    "Panama Drought",
    "Port Strike",
    "Customs Slowdown",
    "Hormuz Tension",
    "Strait of Malacca Tension",
]

synthetic_df = generate_synthetic_shipments(
    n_rows=N_ROWS,
    enable_geopolitical_events=ENABLE_GEOPOLITICAL_EVENTS,
    active_events=ACTIVE_EVENTS,
)

print("Synthetic dataset shape:", synthetic_df.shape)
synthetic_df.head(10)

Synthetic dataset shape: (12000, 24)


,shipment_id,departure_timestamp,origin_port,destination_port,origin_region,destination_region,route_corridor,ocean_side,vessel_type,cargo_type,...,port_handling_days,geopolitical_event,geopolitical_delay_days,expected_delay_days,adjusted_eta_days,risk_score,freight_cost_index,baseline_arrival_timestamp,adjusted_arrival_timestamp,delay_class
0,SHP-0000001,2025-10-30 22:00:00,Shanghai,Houston,East Asia,North America,Pacific Crossing,Pacific Side,Bulk Carrier,Electronics,...,0.89,None,0.00,1.42,22.39,0.269,1.028,2025-11-20 21:15:24.026740490,2025-11-22 07:25:06.079515185,on_time
1,SHP-0000002,2025-02-05 20:00:00,Singapore,Santos,Southeast Asia,South America,Cape Diversion,Indian-Atlantic Side,Tanker,Textiles,...,0.65,None,0.00,0.99,24.13,0.232,1.020,2025-02-28 23:31:36.271944048,2025-03-01 23:11:25.236060987,on_time
2,SHP-0000003,2025-08-13 14:00:00,Shanghai,Los Angeles,East Asia,North America,Pacific Crossing,Pacific Side,Container,Auto Parts,...,1.47,Customs Slowdown,0.53,2.73,13.80,0.417,1.078,2025-08-24 15:38:03.028058946,2025-08-27 09:15:54.984031888,delayed
3,SHP-0000004,2025-08-05 07:00:00,Sydney,Shanghai,Oceania,East Asia,Cape Diversion,Indian-Atlantic Side,Bulk Carrier,Chemicals,...,1.72,None,0.00,1.72,15.69,0.293,1.034,2025-08-19 06:20:09.168756973,2025-08-20 23:32:00.118745810,delayed
4,SHP-0000005,2025-08-27 22:00:00,Houston,Sydney,North America,Oceania,Cape Diversion,Indian-Atlantic Side,Container,Food,...,1.29,None,0.00,1.29,22.64,0.257,1.026,2025-09-18 06:32:50.372501524,2025-09-19 13:26:36.684039105,on_time
5,SHP-0000006,2025-09-19 01:00:00,Rotterdam,Dubai,Europe,Middle East,Suez Corridor,Indian-Atlantic Side,Bulk Carrier,Machinery,...,0.61,Customs Slowdown,0.92,2.73,11.52,0.447,1.095,2025-09-27 19:57:25.794086097,2025-09-30 13:32:27.054349071,delayed
6,SHP-0000007,2025-01-20 10:00:00,Houston,Santos,North America,South America,Panama Corridor,Atlantic-Pacific Side,Bulk Carrier,Textiles,...,1.72,None,0.00,1.89,16.51,0.308,1.038,2025-02-04 00:59:36.423772203,2025-02-05 22:21:14.092409805,delayed
7,SHP-0000008,2025-07-04 10:00:00,Houston,Singapore,North America,Southeast Asia,Pacific Crossing,Pacific Side,Tanker,Auto Parts,...,0.72,Hormuz Tension,1.54,2.26,24.30,0.444,1.115,2025-07-26 10:57:12.022437568,2025-07-28 17:05:21.320431892,delayed
8,SHP-0000009,2025-08-21 10:00:00,Santos,Singapore,South America,Southeast Asia,Cape Diversion,Indian-Atlantic Side,Bulk Carrier,Textiles,...,0.81,None,0.00,1.74,28.68,0.295,1.035,2025-09-17 08:27:52.542914330,2025-09-19 02:17:01.752299350,delayed
9,SHP-0000010,2025-12-13 18:00:00,Singapore,Mumbai,Southeast Asia,South Asia,Cape Diversion,Indian-Atlantic Side,Tanker,Auto Parts,...,1.58,Strait of Malacca Tension,1.85,3.85,11.12,0.583,1.155,2025-12-21 00:22:40.353829034,2025-12-24 20:52:53.860158900,delayed


## 5. Diagnostics

This section summarizes the generated data so you can quickly inspect delay classes, corridor concentration, and event frequency.

In [5]:
# Quick diagnostics
print("\nDelay class distribution:")
print(synthetic_df["delay_class"].value_counts(normalize=True).round(3))

print("\nTop route corridors:")
print(synthetic_df["route_corridor"].value_counts().head())

print("\nEvent usage:")
print(synthetic_df["geopolitical_event"].value_counts().head(10))

synthetic_df[[
    "distance_nm",
    "baseline_eta_days",
    "expected_delay_days",
    "adjusted_eta_days",
    "risk_score",
    "freight_cost_index",
]].describe().T


Delay class distribution:
delay_class
delayed           0.602
on_time           0.328
critical_delay    0.070
Name: proportion, dtype: float64

Top route corridors:
route_corridor
Cape Diversion       8518
Suez Corridor        1101
Pacific Crossing     1071
Atlantic Crossing     785
Panama Corridor       525
Name: count, dtype: int64

Event usage:
geopolitical_event
None                         9470
Customs Slowdown              383
Port Strike                   374
Strait of Malacca Tension     370
Suez Disruption               357
Red Sea Tension               355
Hormuz Tension                353
Panama Drought                338
Name: count, dtype: int64


,count,mean,std,min,25%,50%,75%,max
distance_nm,12000.0,6111.338267,2474.919606,1191.100,4339.400,5906.000,8200.600,11429.900
baseline_eta_days,12000.0,15.717092,7.161793,2.160,10.480,15.030,20.400,43.270
expected_delay_days,12000.0,2.075882,1.071808,0.400,1.330,1.830,2.560,7.040
adjusted_eta_days,12000.0,17.792958,7.215839,2.670,12.500,17.105,22.450,44.940
risk_score,12000.0,0.348655,0.135964,0.183,0.261,0.304,0.376,0.964
freight_cost_index,12000.0,1.058585,0.055208,1.008,1.027,1.037,1.056,1.314


## 6. Export to CSV

This final section writes the processed synthetic dataset to the processed data folder for downstream analysis.

In [6]:
# Save dataset

output_path = Path("../data/processed/synthetic_shipments.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
synthetic_df.to_csv(output_path, index=False)

print(f"Saved: {output_path.resolve()}")

Saved: C:\Users\filip\GitHub\Chain-Pulse-AI\data\processed\synthetic_shipments.csv
